In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.models import Model
import random

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.19.0


In [ ]:
# Define vocabulary tokens
tokens = ['<pad>', '<sos>', '<eos>'] + [str(i) for i in range(10)]

token_to_id = {t:i for i,t in enumerate(tokens)}
id_to_token = {i:t for t,i in token_to_id.items()}

vocab_size = len(tokens)

print("Vocabulary:", tokens)
print("Vocabulary Size:", vocab_size)

Vocabulary: ['<pad>', '<sos>', '<eos>', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
Vocabulary Size: 13


In [ ]:
# Generate random sequence
def generate_sequence():
    length = random.randint(3,6)
    return [str(random.randint(0,9)) for _ in range(length)]


N = 3000

encoder_text = []
decoder_input_text = []
decoder_output_text = []

for _ in range(N):

    seq = generate_sequence()

    encoder_text.append(seq)

    decoder_input_text.append(['<sos>'] + seq)

    decoder_output_text.append(seq + ['<eos>'])

In [ ]:
max_encoder_length = max(len(seq) for seq in encoder_text)
max_decoder_length = max(len(seq) for seq in decoder_input_text)


def vectorize(sequences, max_len):

    data = np.zeros((len(sequences), max_len), dtype="int32")

    for i, seq in enumerate(sequences):

        for j, token in enumerate(seq):

            data[i, j] = token_to_id[token]

    return data


encoder_input_data = vectorize(encoder_text, max_encoder_length)

decoder_input_data = vectorize(decoder_input_text, max_decoder_length)

decoder_output_data = vectorize(decoder_output_text, max_decoder_length)


print("Encoder data shape:", encoder_input_data.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder output shape:", decoder_output_data.shape)

Encoder data shape: (3000, 6)
Decoder input shape: (3000, 7)
Decoder output shape: (3000, 7)


In [ ]:
embedding_dim = 64
latent_dim = 128


# Shared embedding layer
embedding_layer = Embedding(vocab_size, embedding_dim, mask_zero=True)


# ---------------- Encoder ----------------

encoder_inputs = Input(shape=(None,))

encoder_embedding = embedding_layer(encoder_inputs)

_, state_h, state_c = LSTM(latent_dim, return_state=True)(encoder_embedding)

encoder_states = [state_h, state_c]


# ---------------- Decoder ----------------

decoder_inputs = Input(shape=(None,))

decoder_embedding = embedding_layer(decoder_inputs)

decoder_outputs, _, _ = LSTM(

    latent_dim,
    return_sequences=True,
    return_state=True

)(decoder_embedding, initial_state=encoder_states)


decoder_dense = Dense(vocab_size, activation="softmax")

decoder_outputs = decoder_dense(decoder_outputs)


# ---------------- Model ----------------

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(

    optimizer="adam",
    loss="sparse_categorical_crossentropy"
)


model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 64)  │        832 │ input_layer[0][0… │
│ (Embedding)         │                   │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 128),     │     98,816 │ embedding[0][0],  │
│                     │ (None, 128),      │            │ not_equal[0][0]   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │     98,816 │ embedding[1][0],  │
│                     │ 128), (None,      │            │ lstm[0][1],       │
│                     │ 128), (None,      │            │ lstm[0][2]        │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 13)  │      1,677 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 200,141 (781.80 KB)

 Trainable params: 200,141 (781.80 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(

    [encoder_input_data, decoder_input_data],
    decoder_output_data,

    batch_size=64,
    epochs=15,
    validation_split=0.1

)

Epoch 1/15
43/43 ━━━━━━━━━━━━━━━━━━━━ 8s 48ms/step - loss: 2.4368 - val_loss: 1.9179
Epoch 2/15
43/43 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - loss: 1.7880 - val_loss: 1.3947
Epoch 3/15
43/43 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - loss: 1.2749 - val_loss: 0.9279
Epoch 4/15
43/43 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - loss: 0.8334 - val_loss: 0.6021
Epoch 5/15
43/43 ━━━━━━━━━━━━━━━━━━━━ 3s 42ms/step - loss: 0.5286 - val_loss: 0.3983
Epoch 6/15
43/43 ━━━━━━━━━━━━━━━━━━━━ 3s 66ms/step - loss: 0.3425 - val_loss: 0.3119
Epoch 7/15
43/43 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - loss: 0.2497 - val_loss: 0.2359
Epoch 8/15
43/43 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - loss: 0.1790 - val_loss: 0.1715
Epoch 9/15
43/43 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - loss: 0.1347 - val_loss: 0.1443
Epoch 10/15
43/43 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - loss: 0.1142 - val_loss: 0.1171
Epoch 11/15
43/43 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - loss: 0.0806 - val_loss: 0.0895
Epoch 12/15
43/43 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - loss: 0.0

In [ ]:
encoder_model = Model(encoder_inputs, encoder_states)

In [ ]:
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))

decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]


decoder_embedding2 = embedding_layer(decoder_inputs)

decoder_outputs2, state_h2, state_c2 = LSTM(

    latent_dim,
    return_sequences=True,
    return_state=True

)(decoder_embedding2, initial_state=decoder_states_inputs)


decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(

    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2, state_h2, state_c2]

)

In [ ]:
def decode_sequence(input_seq):

    states = encoder_model.predict(input_seq)

    target_seq = np.array([[token_to_id["<sos>"]]])

    decoded = []

    while True:

        output_tokens, h, c = decoder_model.predict([target_seq] + states)

        sampled_index = np.argmax(output_tokens[0, -1, :])

        sampled_token = id_to_token[sampled_index]

        if sampled_token == "<eos>" or len(decoded) > max_decoder_length:
            break

        decoded.append(sampled_token)

        target_seq = np.array([[sampled_index]])

        states = [h, c]

    return decoded

In [ ]:
for i in range(10):

    input_seq = encoder_input_data[i:i+1]

    decoded = decode_sequence(input_seq)

    input_tokens = [id_to_token[idx] for idx in input_seq[0] if idx != 0]

    print("Input :", input_tokens)
    print("Output:", decoded)
    print()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 359ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 383ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Input : ['5', '7', '8', '3', '2']
Output: ['5', '5', '5', '5', '5', '5', '3', '3']

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Input : ['5', '7', '3', '5', '2']
Output: ['5', '5', '5', '5', '5', '3', '3', '3']

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms